# BTCUSDT London Volatility Breakout

Five-year Python backtest. One-minute Binance data. One trade per London weekday.

## 1. Load the saved result

Raw data is excluded from Git. The repository includes the resulting trade and equity files.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent
trades = pd.read_csv(ROOT / 'results' / 'trades.csv', parse_dates=['entry_time_utc', 'exit_time_utc'])
equity = pd.read_csv(ROOT / 'results' / 'daily_equity.csv', index_col=0, parse_dates=True)['equity_usd']
summary = json.loads((ROOT / 'results' / 'summary.json').read_text())
summary['metrics']

## 2. Equity and drawdown

![Equity and drawdown](results/equity_drawdown.png)

In [ ]:
drawdown = equity / equity.cummax() - 1
pd.Series({
    'ending_equity': equity.iloc[-1],
    'return': equity.iloc[-1] / equity.iloc[0] - 1,
    'max_drawdown': drawdown.min(),
    'trades': len(trades),
})

## 3. Position sizing

Risk is 0.05% of current equity at every entry. BTC quantity changes with both equity and the stop distance.

In [ ]:
trades[['initial_risk_usd', 'units', 'pnl_usd', 'ending_equity_usd']].describe()

## 4. Yearly result

![Yearly performance](results/yearly_performance.png)

In [ ]:
yearly = trades.assign(year=trades.entry_time_utc.dt.year).groupby('year').agg(
    trades=('pnl_usd', 'size'),
    pnl_usd=('pnl_usd', 'sum'),
    win_rate=('pnl_usd', lambda values: (values > 0).mean()),
)
yearly

## 5. Conclusion

The zero-cost baseline returned 2.63% with a 1.34% maximum drawdown. The profit factor was 1.11. Most gains came from 2024 and 2025.

The result is weak after considering omitted costs. It shows the research process, not a live-trading claim.